# MobileNetV2 on CIFAR-10, FP32 Training + INT8 Quantization

This notebook trains a MobileNetV2 on CIFAR-10 in FP32, then applies post-training static quantization (INT8) and compares accuracy, inference speed, and model size.

## 0. Environment Check

In [1]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device    : {DEVICE}")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU    : Tesla T4
Using device    : cuda


## 1. Data CIFAR-10

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

trainset = datasets.CIFAR10(root="./data", train=True,  download=True,  transform=transform_train)
testset  = datasets.CIFAR10(root="./data", train=False, download=True,  transform=transform_test)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
testloader  = DataLoader(testset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(trainloader)} | Test batches: {len(testloader)}")

100%|██████████| 170M/170M [00:04<00:00, 41.1MB/s] 


Train batches: 782 | Test batches: 157


## 2. Model MobileNetV2

In [3]:
import torch.nn as nn
from torchvision.models import mobilenet_v2

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = mobilenet_v2(weights=None, num_classes=10)

    def forward(self, x):
        return self.model(x)

print("CNN (MobileNetV2) defined.")

CNN (MobileNetV2) defined.


## 3. Evaluation Helper

In [4]:
def evaluate(model, loader, device=None):
    """Returns top-1 accuracy. Moves batches to `device` if provided."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            if device:
                x, y = x.to(device), y.to(device)
            out  = model(x)
            pred = out.argmax(dim=1)
            total   += y.size(0)
            correct += (pred == y).sum().item()
    return correct / total

In [ ]:
import time, os

model = CNN().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # scheduler.step()
    print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  loss: {total_loss:.4f}")


torch.save(model.state_dict(), "fp32.pth")
print("\nModel saved to fp32.pth")
print(f"Saved at: {os.path.abspath('fp32.pth')}")